In [ ]:
# Define prompt strings as constants
DESCRIPTION_PROMPT = [
    ("system", """Given the JSON example(s) for a task type:
     
{raw_example}

Provide a concise description of the task type, including the format and style
of the input and output. If there are multiple examples, provide an overall
description and ignore unique parts.

Format your response as follows:
Task Description: [Your description here]
""")
]

# INPUT_ANALYSIS_PROMPT = [
#     ("system", """Describe input dimensions and attributes for a specific task type.
# Provide names, very brief descriptions, ranges, typical values, extreme values and
# examples for each.

# Format your response as follows:
# Input Analysis: [Your analysis here]
# """),
#     ("user", """Task Description:

# {description}

# """)
# ]

INPUT_ANALYSIS_PROMPT = [
    ("system", """For the specific task type, analyze the possible task inputs across multiple dimensions.
     
Conduct a detailed analysis and enumerate:

1. Core Attributes: Identify the fundamental properties or characteristics of this input type.
1. Variation Dimensions: For each dimension that may vary, specify:
   - Dimension name
   - Possible range of values or options
   - Impact on input nature or task difficulty
1. Constraints: List any rules or limitations that must be adhered to.
1. Edge Cases: Describe extreme or special scenarios that may test the robustness of task processing.
1. External Factors: Enumerate factors that might influence input generation or task completion.
1. Potential Extensions: Propose ways to expand or modify this input type to create new variants.

Format your response as follows:
Input Analysis: [Your analysis here]
"""),
    ("user", """Task Description:

{description}

""")
]

BRIEFS_PROMPT = [
    ("system", """Given the task type description, and input analysis, generate
descriptions for {generating_batch_size} new examples with detailed attributes
based on this task type. But don't provide any detailed task output.

Use the input analysis to create diverse and comprehensive example briefs that
cover various input dimensions and attribute ranges.

Format your response as a valid YAML object with a single key 'new_example_briefs'
containing a YAML array of {generating_batch_size} objects, each with a
'example_brief' field.
"""),
    ("user", """Task Description:

{description}

Input Analysis:

{input_analysis}

""")
]

EXAMPLES_FROM_BRIEFS_PROMPT = [
    ("system", """Given the task type description, brief descriptions for new examples, 
and JSON example(s), generate {generating_batch_size} more input/output examples for this task type,
strictly based on the brief descriptions. Ensure that the new examples are
consistent with the brief descriptions and do not introduce any new information
not present in the briefs.

Format your response as a valid JSON object with a single key 'examples' 
containing a JSON array of {generating_batch_size} objects, each with 'input' and 'output' fields.
"""),
    ("user", """Task Description:

{description}

New Example Briefs: 

{new_example_briefs}

Example(s):

{raw_example}

""")
]

EXAMPLES_PROMPT = [
    ("system", """Given the task type description, and input/output example(s), generate {generating_batch_size}
new input/output examples for this task type.

Format your response as a valid JSON object with a single key 'examples' 
containing a JSON array of {generating_batch_size} objects, each with 'input' and 'output' fields.
"""),
    ("user", """Task Description:

{description}

Example(s):

{raw_example}

""")
]


In [ ]:
import json
import yaml
from langchain.prompts import ChatPromptTemplate
from langchain.chat_models import ChatOpenAI
from langchain.schema.output_parser import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from langchain_core.output_parsers import JsonOutputParser
from langchain.output_parsers import YamlOutputParser


class TaskDescriptionGenerator:
    def __init__(self, model):        
        self.description_prompt = ChatPromptTemplate.from_messages(DESCRIPTION_PROMPT)
        self.input_analysis_prompt = ChatPromptTemplate.from_messages(INPUT_ANALYSIS_PROMPT)
        self.briefs_prompt = ChatPromptTemplate.from_messages(BRIEFS_PROMPT)
        self.examples_from_briefs_prompt = ChatPromptTemplate.from_messages(EXAMPLES_FROM_BRIEFS_PROMPT)
        self.examples_prompt = ChatPromptTemplate.from_messages(EXAMPLES_PROMPT)

        json_model = model.bind(response_format={"type": "json_object"})

        output_parser = StrOutputParser()
        json_parse = JsonOutputParser()

        self.description_chain = self.description_prompt | model | output_parser
        self.input_analysis_chain = self.input_analysis_prompt | model | output_parser
        self.briefs_chain = self.briefs_prompt | model | output_parser
        self.examples_from_briefs_chain = self.examples_from_briefs_prompt | json_model | json_parse
        self.examples_chain = self.examples_prompt | json_model | json_parse

        self.chain = (
            RunnablePassthrough.assign(raw_example = lambda x: json.dumps(x["example"], ensure_ascii=False))
            | RunnablePassthrough.assign(description = self.description_chain)
            | {
                "description": lambda x: x["description"],
                "examples_from_briefs": RunnablePassthrough.assign(input_analysis = self.input_analysis_chain)
                    | RunnablePassthrough.assign(new_example_briefs = self.briefs_chain) 
                    | RunnablePassthrough.assign(examples = self.examples_from_briefs_chain | (lambda x: x["examples"])),
                "examples": self.examples_chain
            }
            | RunnablePassthrough.assign(
                additional_examples=lambda x: (
                    list(x["examples_from_briefs"]["examples"])
                    + list(x["examples"]["examples"])
                )
            )
        )

    def process(self, input_str, generating_batch_size=3):
        try:
            try:
                example_dict = json.loads(input_str)
            except ValueError:
                try:
                    example_dict = yaml.safe_load(input_str)
                except yaml.YAMLError as e:
                    raise ValueError("Invalid input format. Expected a JSON or YAML object.") from e

            # If example_dict is a list, filter out invalid items
            if isinstance(example_dict, list):
                example_dict = [item for item in example_dict if isinstance(item, dict) and 'input' in item and 'output' in item]

            # If example_dict is not a list, check if it's a valid dict
            elif not isinstance(example_dict, dict) or 'input' not in example_dict or 'output' not in example_dict:
                raise ValueError("Invalid input format. Expected an object with 'input' and 'output' fields.")

            # Move the original content to a key named 'example'
            input_dict = {"example": example_dict, "generating_batch_size": generating_batch_size}

            # Invoke the chain with the parsed input dictionary
            result = self.chain.invoke(input_dict)
            return result

        except Exception as e:
            raise RuntimeError(f"An error occurred during processing: {str(e)}")

    def generate_description(self, input_str):
        return self.description_chain.invoke(input_str)

    def analyze_input(self, description):
        return self.input_analysis_chain.invoke(description)

    def generate_briefs(self, description, input_analysis, generating_batch_size):
        return self.briefs_chain.invoke({
            "description": description,
            "input_analysis": input_analysis,
            "generating_batch_size": generating_batch_size
        })

    def generate_examples_from_briefs(self, description, new_example_briefs, raw_example, generating_batch_size):
        return self.examples_from_briefs_chain.invoke({
            "description": description,
            "new_example_briefs": new_example_briefs,
            "raw_example": raw_example,
            "generating_batch_size": generating_batch_size
        })

    def generate_examples(self, description, raw_example, generating_batch_size):
        return self.examples_chain.invoke({
            "description": description,
            "raw_example": raw_example,
            "generating_batch_size": generating_batch_size
        })

In [ ]:
import gradio as gr

def process_json(input_json, model_name, generating_batch_size, temperature):
    try:
        model = ChatOpenAI(model=model_name, temperature=temperature, max_retries=3)
        generator = TaskDescriptionGenerator(model)
        result = generator.process(input_json, generating_batch_size)
        description = result["description"]
        input_analysis = result["examples_from_briefs"]["input_analysis"]
        examples = [[example["input"], example["output"]] for example in result["additional_examples"]]
        return description, input_analysis, examples
    except Exception as e:
        raise gr.Error(f"An error occurred: {str(e)}")
    
def generate_description_only(input_json, model_name, temperature):
    try:
        model = ChatOpenAI(model=model_name, temperature=temperature, max_retries=3)
        generator = TaskDescriptionGenerator(model)
        description = generator.generate_description(input_json)
        return description
    except Exception as e:
        raise gr.Error(f"An error occurred: {str(e)}")

def analyze_input(description, model_name, temperature):
    try:
        model = ChatOpenAI(model=model_name, temperature=temperature, max_retries=3)
        generator = TaskDescriptionGenerator(model)
        input_analysis = generator.analyze_input(description)
        return input_analysis
    except Exception as e:
        raise gr.Error(f"An error occurred: {str(e)}")
    
def generate_briefs(description, input_analysis, generating_batch_size, model_name, temperature):
    try:
        model = ChatOpenAI(model=model_name, temperature=temperature, max_retries=3)
        generator = TaskDescriptionGenerator(model)
        briefs = generator.generate_briefs(description, input_analysis, generating_batch_size)
        return briefs
    except Exception as e:
        raise gr.Error(f"An error occurred: {str(e)}")
    
def generate_examples_from_briefs(description, new_example_briefs, raw_example, generating_batch_size, model_name, temperature):
    try:
        model = ChatOpenAI(model=model_name, temperature=temperature, max_retries=3)
        generator = TaskDescriptionGenerator(model)
        examples = generator.generate_examples_from_briefs(description, new_example_briefs, raw_example, generating_batch_size)
        return examples
    except Exception as e:
        raise gr.Error(f"An error occurred: {str(e)}")
    
def generate_examples(description, raw_example, generating_batch_size, model_name, temperature):
    try:
        model = ChatOpenAI(model=model_name, temperature=temperature, max_retries=3)
        generator = TaskDescriptionGenerator(model)
        examples = generator.generate_examples(description, raw_example, generating_batch_size)
        return examples
    except Exception as e:
        raise gr.Error(f"An error occurred: {str(e)}")

def format_selected_example(evt: gr.SelectData, examples):
    if evt.index[0] < len(examples):
        selected_example = examples.iloc[evt.index[0]]  # Use iloc to access by integer position
        json_example = json.dumps({"input": selected_example.iloc[0], "output": selected_example.iloc[1]}, indent=2, ensure_ascii=False)
        return json_example
    return ""

with gr.Blocks(title="Task Description Generator") as demo:
    gr.Markdown("# Task Description Generator")
    gr.Markdown("Enter a JSON object with 'input' and 'output' fields to generate a task description and additional examples.")

    with gr.Row():
        with gr.Column(scale=1):  # Inputs column
            input_json = gr.Textbox(label="Input JSON", lines=10, show_copy_button=True)
            model_name = gr.Dropdown(
                label="Model Name",
                choices=["llama3-70b-8192", "llama3-8b-8192", "llama-3.1-70b-versatile", "llama-3.1-8b-instant", "gemma2-9b-it"],
                value="llama3-70b-8192"
            )
            temperature = gr.Slider(label="Temperature", value=1.0, minimum=0.0, maximum=1.0, step=0.1)
            generating_batch_size = gr.Slider(label="Generating Batch Size", value=3, minimum=1, maximum=10, step=1)
            with gr.Row():
                submit_button = gr.Button("Generate", variant="primary")
                generate_description_button = gr.Button("Generate Description", variant="secondary")

        with gr.Column(scale=1):  # Outputs column
            description_output = gr.Textbox(label="Description", lines=5, show_copy_button=True)
            analyze_input_button = gr.Button("Analyze Input", variant="secondary")
            input_analysis_output = gr.Textbox(label="Input Analysis", lines=5, show_copy_button=True)
            generate_briefs_button = gr.Button("Generate Briefs", variant="secondary")
            example_briefs_output = gr.Textbox(label="Example Briefs", lines=5, show_copy_button=True)
            generate_examples_from_briefs_button = gr.Button("Generate Examples from Briefs", variant="secondary")
            examples_from_briefs_output = gr.DataFrame(label="Examples from Briefs", headers=["Input", "Output"], interactive=False)
            examples_output = gr.DataFrame(label="Examples", headers=["Input", "Output"], interactive=False)
            new_example_json = gr.Textbox(label="New Example JSON", lines=5, show_copy_button=True)

            clear_button = gr.ClearButton([input_json, description_output, input_analysis_output,
                                           example_briefs_output, examples_from_briefs_output,
                                           examples_output, new_example_json])

    submit_button.click(
        fn=process_json,
        inputs=[input_json, model_name, generating_batch_size, temperature],
        outputs=[description_output, input_analysis_output, examples_output]
    )

    analyze_input_button.click(
        fn=analyze_input,
        inputs=[description_output, model_name, temperature],
        outputs=[input_analysis_output]
    )

    generate_description_button.click(
        fn=generate_description_only,
        inputs=[input_json, model_name, temperature],
        outputs=[description_output]
    )

    generate_briefs_button.click(
        fn=generate_briefs,
        inputs=[description_output, input_analysis_output, generating_batch_size, model_name, temperature],
        outputs=[example_briefs_output]
    )

    generate_examples_from_briefs_button.click(
        fn=generate_examples_from_briefs,
        inputs=[description_output, example_briefs_output, input_json, generating_batch_size, model_name, temperature],
        outputs=[examples_from_briefs_output]
    )

    examples_output.select(
        fn=format_selected_example,
        inputs=[examples_output],
        outputs=[new_example_json]
    )

    gr.Markdown("### Manual Flagging")
    with gr.Row():
        flag_button = gr.Button("Flag")
        flag_reason = gr.Textbox(label="Reason for flagging")

    flagging_callback = gr.CSVLogger()
    flag_button.click(
        lambda *args: flagging_callback.flag(args),
        inputs=[input_json, model_name, generating_batch_size, description_output, examples_output, flag_reason],
        outputs=[]
    )

if __name__ == "__main__":
    demo.launch()